# 43 — Adaptive Resource Allocation

**Engineering statement:** Operating regimes specify adaptive resource allocation.

Notebook 37 closes the first major arc: local mechanisms become operating regimes.

Notebook 43 opens the second major arc: operating regimes become controller policies. The serving system estimates state, classifies the active regime, partitions scarce resources, schedules verification, observes throughput, and updates allocation.


## Start here

```text
operating regime
→ adaptive allocator
→ budget partition
→ verification scheduler
→ serving system
→ observed throughput
→ feedback update
```

Notebook 37 asks:

> Which operating regime is active?

Notebook 43 asks:

> Given that regime, how should limited resources be allocated?


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_adaptive_resource_allocation"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12, lw=1.5):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=lw,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, style="->", lw=1.8, rad=0.0):
    ax.annotate(
        "", xy=end, xytext=start,
        arrowprops=dict(arrowstyle=style, lw=lw, connectionstyle=f"arc3,rad={rad}")
    )


## 1. Repository arc

The first arc moves from verification mechanisms to operating-regime classification.

The second arc moves from operating-regime classification to adaptive controllers, infrastructure, and distributed scheduling.


In [ ]:
# 43_repository_arc.png

fig, ax = plt.subplots(figsize=(13.8, 6.0))
ax.axis("off")
ax.set_xlim(0, 13.8)
ax.set_ylim(0, 6)

first_arc = [
    ("00\nContext", 0.3, 4.1, 1.18, 0.72),
    ("07\nVerification\nResource", 2.0, 4.1, 1.28, 0.72),
    ("13\nConfidence\nScheduling", 3.85, 4.1, 1.38, 0.72),
    ("17\nSemi-AR\nDrafting", 5.85, 4.1, 1.28, 0.72),
    ("23\nThroughput\nObjective", 7.65, 4.1, 1.38, 0.72),
    ("29\nHardware\nConstraints", 9.65, 4.1, 1.28, 0.72),
    ("37\nOperating\nRegimes", 11.35, 4.1, 1.28, 0.72),
]
for label, x, y, w, h in first_arc:
    rounded_box(ax, (x, y), w, h, label, fontsize=10)
for i in range(len(first_arc)-1):
    _, x, y, w, h = first_arc[i]
    _, x2, y2, w2, h2 = first_arc[i+1]
    arrow(ax, (x+w+0.05, y+h/2), (x2-0.05, y2+h2/2))

ax.plot([0.65, 12.3], [3.42, 3.42], color="black", lw=1.2)
ax.text(6.6, 3.58, "first arc: mechanism → operating regime", ha="center", fontsize=12)

second_arc = [
    ("43\nAdaptive Resource\nAllocation", 4.0, 1.65, 1.55, 0.76),
    ("49\nAdaptive\nInfrastructure", 6.1, 1.65, 1.45, 0.76),
    ("53\nDistributed\nScheduling", 8.1, 1.65, 1.45, 0.76),
    ("59\nCluster\nOptimization", 10.05, 1.65, 1.45, 0.76),
]
for label, x, y, w, h in second_arc:
    rounded_box(ax, (x, y), w, h, label, fontsize=10)
for i in range(len(second_arc)-1):
    _, x, y, w, h = second_arc[i]
    _, x2, y2, w2, h2 = second_arc[i+1]
    arrow(ax, (x+w+0.05, y+h/2), (x2-0.05, y2+h2/2))

ax.text(7.1, 0.85, "second arc: controller → infrastructure → distributed scheduling", ha="center", fontsize=12)
ax.set_title("Resource allocation opens the second systems arc", fontsize=22, pad=14)
savefig("43_repository_arc.png")


## 2. Hero controller architecture

This is the central Notebook 43 figure.

The controller does not directly optimize one scalar. It estimates the serving state, classifies the regime, selects a resource policy, partitions the budget, schedules verification, observes throughput, and updates.


In [ ]:
# 43_controller_architecture.png

fig, ax = plt.subplots(figsize=(12.5, 7.3))
ax.axis("off")
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 7.3)

# Main vertical controller path
rounded_box(ax, (1.0, 5.7), 2.0, 0.72, "Incoming\nrequests", fontsize=11)
rounded_box(ax, (4.9, 5.7), 2.0, 0.72, "State\nestimator", fontsize=11)
rounded_box(ax, (4.55, 4.55), 2.7, 0.72, "Operating-regime\nclassifier", fontsize=11)
rounded_box(ax, (4.85, 3.4), 2.1, 0.72, "Adaptive\nallocator", fontsize=11)

arrow(ax, (3.05, 6.06), (4.85, 6.06))
arrow(ax, (5.9, 5.67), (5.9, 5.32))
arrow(ax, (5.9, 4.52), (5.9, 4.17))

# Budget branches
budget_y = 2.25
budgets = [
    ("compute", 1.0, budget_y, 1.35, 0.6),
    ("memory", 3.0, budget_y, 1.35, 0.6),
    ("verify", 5.0, budget_y, 1.35, 0.6),
    ("reserve", 7.0, budget_y, 1.35, 0.6),
    ("latency", 9.0, budget_y, 1.35, 0.6),
]
for label, x, y, w, h in budgets:
    rounded_box(ax, (x, y), w, h, label, fontsize=10)
    arrow(ax, (5.9, 3.37), (x+w/2, y+h+0.05), lw=1.5)

rounded_box(ax, (4.65, 0.95), 2.5, 0.75, "Serving\nsystem", fontsize=12)
for _, x, y, w, h in budgets:
    arrow(ax, (x+w/2, y-0.05), (5.9, 1.75), lw=1.3)

rounded_box(ax, (8.9, 0.95), 2.15, 0.75, "Observed\nsystem state", fontsize=11)
arrow(ax, (7.2, 1.33), (8.85, 1.33))

# Feedback loop
arrow(ax, (9.95, 1.75), (9.95, 6.05), lw=1.6, rad=0.0)
arrow(ax, (9.95, 6.05), (6.95, 6.05), lw=1.6)
arrow(ax, (1.98, 5.66), (1.98, 3.25), lw=1.4)
rounded_box(ax, (0.95, 2.45), 2.05, 0.65, "State\nupdate", fontsize=10)
arrow(ax, (8.9, 1.33), (3.05, 2.77), lw=1.4, rad=0.18)
arrow(ax, (1.98, 3.14), (1.98, 5.66), lw=1.4)

ax.text(6.25, 0.35, "Operating regimes specify adaptive resource allocation.", ha="center", fontsize=13)
ax.set_title("Adaptive resource-allocation controller", fontsize=22, pad=14)
savefig("43_controller_architecture.png")


## 3. Resource allocation flow

A compact flow for the same control idea:

\[
\rho_t \rightarrow B_t \rightarrow a_t \rightarrow \ell_t \rightarrow \Theta_t
\]

where \(\rho_t\) is the regime, \(B_t\) is the budget, \(a_t\) is the allocation, \(\ell_t\) is the scheduled verification decision, and \(\Theta_t\) is observed throughput.


In [ ]:
# 43_resource_flow.png

fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.axis("off")
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4)

nodes = [
    ("Operating\nregime", 0.7, 2.15, 1.5, 0.78),
    ("Adaptive\ncontroller", 3.0, 2.15, 1.55, 0.78),
    ("Budget\npartition", 5.35, 2.15, 1.55, 0.78),
    ("Verification\nscheduler", 7.7, 2.15, 1.7, 0.78),
    ("Serving\nsystem", 10.2, 2.15, 1.45, 0.78),
]
for label, x, y, w, h in nodes:
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
for i in range(len(nodes)-1):
    _, x, y, w, h = nodes[i]
    _, x2, y2, w2, h2 = nodes[i+1]
    arrow(ax, (x+w+0.06, y+h/2), (x2-0.06, y2+h2/2))

rounded_box(ax, (3.3, 0.75), 5.9, 0.62, "The controller spends scarce resources according to the active regime.", fontsize=12)
ax.set_title("Resource allocation flow for adaptive serving", fontsize=22, pad=14)
savefig("43_resource_flow.png")


## 4. Budget partition

The budget vector is simplified into four resource categories:

\[
B_t=(B_{\rm compute},B_{\rm memory},B_{\rm verify},B_{\rm reserve})
\]

Reserve includes latency headroom, admission slack, and fail-safe capacity.


In [ ]:
# 43_budget_partition_v2.png

regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
categories = ["compute", "memory", "verify", "reserve"]

budgets = np.array([
    [0.30, 0.25, 0.30, 0.15],
    [0.22, 0.30, 0.20, 0.28],
    [0.32, 0.18, 0.25, 0.25],
    [0.25, 0.22, 0.18, 0.35],
])

fig, ax = plt.subplots(figsize=(10.7, 5.8))
left = np.zeros(len(regimes))
y = np.arange(len(regimes))

for i, cat in enumerate(categories):
    ax.barh(y, budgets[:, i], left=left, label=cat)
    for j, val in enumerate(budgets[:, i]):
        ax.text(left[j] + val/2, j, f"{cat}\n{val:.2f}", ha="center", va="center", fontsize=10)
    left += budgets[:, i]

ax.set_yticks(y, regimes)
ax.set_xlim(0, 1.0)
ax.set_xlabel("normalized resource budget")
ax.set_title("Adaptive budget partitions by operating regime", fontsize=22)
ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.12), loc="upper center")
ax.grid(axis="x", alpha=0.25)
savefig("43_budget_partition.png")

(RESULTS / "43_budget_partition_v2.json").write_text(
    json.dumps({"regimes": regimes, "categories": categories, "budgets": budgets.tolist()}, indent=2),
    encoding="utf-8"
)


## 5. Policy curves

Instead of only measuring throughput gain, a controller figure should show how policy changes as resource budget changes.

For each operating regime, the allocation fraction shifts among compute, verification, memory, and reserve.


In [ ]:
# 43_policy_curves.png

x = np.linspace(0.05, 1.0, 80)

policy_curves = {
    "compute fraction": 0.20 + 0.55*(1 - np.exp(-2.0*x)),
    "verification fraction": 0.18 + 0.42*(1 - np.exp(-3.0*x)) - 0.08*np.maximum(x-0.72, 0),
    "memory fraction": 0.36 - 0.12*(1 - np.exp(-2.5*x)) + 0.08*np.exp(-((x-0.35)/0.22)**2),
    "reserve fraction": 0.34*np.exp(-2.5*x) + 0.10,
}

fig, ax = plt.subplots(figsize=(9.5, 6.0))
for label, vals in policy_curves.items():
    ax.plot(x, vals, marker="o", markevery=9, label=label)

ax.set_xlabel("available resource budget")
ax.set_ylabel("allocation fraction")
ax.set_ylim(0, 0.85)
ax.set_title("Adaptive policy curves allocate budget by regime", fontsize=22)
ax.grid(True, alpha=0.3)
ax.legend()
savefig("43_policy_curves.png")


## 6. Controller policy surface

A controller should use measurable serving variables.

This map uses GPU utilization and queue depth as the axes. The selected policy changes across regions.


In [ ]:
# 43_policy_surface.png

gpu = np.linspace(0.1, 1.0, 130)
queue = np.linspace(0.1, 1.0, 120)
G, Q = np.meshgrid(gpu, queue)

# Policy labels:
# 0 balanced, 1 compute-limited, 2 memory-limited, 3 latency-protected
score_bal = np.exp(-((G-0.55)/0.22)**2 - ((Q-0.45)/0.22)**2)
score_compute = 0.9*G + 0.25*Q - 0.45
score_memory = 0.75*G + 0.55*Q - 0.65 + 0.25*np.exp(-((G-0.75)/0.18)**2)
score_latency = 1.25*Q - 0.30*G - 0.35

scores = np.stack([score_bal, score_compute, score_memory, score_latency])
policy = np.argmax(scores, axis=0)

fig, ax = plt.subplots(figsize=(9.3, 6.5))
im = ax.imshow(
    policy,
    origin="lower",
    aspect="auto",
    extent=[gpu.min(), gpu.max(), queue.min(), queue.max()],
    interpolation="nearest"
)
ax.contour(G, Q, policy, levels=[0.5, 1.5, 2.5], linewidths=0.9, alpha=0.6)
ax.set_xlabel("GPU utilization")
ax.set_ylabel("queue depth")
ax.set_title("Controller policy surface from measurable serving state", fontsize=22)

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["balanced", "compute-limited", "memory-limited", "latency-protected"])
cbar.set_label("selected policy")

ax.text(0.43, 0.34, "balanced", fontsize=12, ha="center")
ax.text(0.78, 0.30, "compute-\nlimited", fontsize=12, ha="center")
ax.text(0.78, 0.70, "memory-\nlimited", fontsize=12, ha="center")
ax.text(0.35, 0.82, "latency-\nprotected", fontsize=12, ha="center")
savefig("43_policy_surface.png")


## 7. Controller phase map

This abstract map is less about a numeric surface and more about controller interpretation.

The axes describe resource pressure directions. The center is balanced; the quadrants identify the policy tendency.


In [ ]:
# 43_phase_map.png

fig, ax = plt.subplots(figsize=(8.6, 7.2))
ax.axis("off")
ax.set_xlim(-1.25, 1.25)
ax.set_ylim(-1.25, 1.25)

# Axes
arrow(ax, (-1.05, 0), (1.05, 0), lw=2.0)
arrow(ax, (0, -1.05), (0, 1.05), lw=2.0)

ax.text(1.12, 0.02, "compute\navailability", ha="left", va="center", fontsize=13, weight="bold")
ax.text(-1.12, 0.02, "compute\npressure", ha="right", va="center", fontsize=13, weight="bold")
ax.text(0.0, 1.13, "memory\npressure", ha="center", va="bottom", fontsize=13, weight="bold")
ax.text(0.0, -1.13, "latency\nreserve", ha="center", va="top", fontsize=13, weight="bold")

rounded_box(ax, (-0.35, -0.16), 0.7, 0.32, "balanced", fontsize=12)
rounded_box(ax, (-0.95, 0.58), 0.62, 0.34, "memory +\ncompute", fontsize=10)
rounded_box(ax, (0.35, 0.58), 0.62, 0.34, "memory +\nverification", fontsize=10)
rounded_box(ax, (-0.95, -0.88), 0.62, 0.34, "latency +\ncompute", fontsize=10)
rounded_box(ax, (0.35, -0.88), 0.62, 0.34, "latency +\nconfidence", fontsize=10)

ax.set_title("Controller map for adaptive resource allocation", fontsize=22, pad=14)
savefig("43_phase_map.png")


## 8. Resource allocation surface

This surface remains useful as a diagnostic: compute and memory availability jointly determine a throughput proxy.

The policy points are examples of where different allocations might operate.


In [ ]:
# 43_resource_surface.png

compute = np.linspace(0.1, 1.0, 120)
memory = np.linspace(0.1, 1.0, 110)
C, M = np.meshgrid(compute, memory)

throughput = (1 - np.exp(-3.1*C)) * (1 - np.exp(-2.9*M))
throughput *= (1 - 0.14*np.abs(C-M))
throughput *= np.exp(-0.25*np.maximum(C-0.88, 0)**2 - 0.30*np.maximum(M-0.88, 0)**2)

fig, ax = plt.subplots(figsize=(9.3, 6.6))
im = ax.imshow(
    throughput,
    aspect="auto",
    origin="lower",
    extent=[compute.min(), compute.max(), memory.min(), memory.max()],
    interpolation="bilinear"
)
ax.contour(C, M, throughput, levels=9, linewidths=0.8, alpha=0.45)
ax.set_xlabel("compute budget")
ax.set_ylabel("memory budget")
ax.set_title("Resource allocation surface for adaptive serving", fontsize=22)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("throughput proxy")

points = {
    "balanced": (0.62, 0.62),
    "compute-limited": (0.42, 0.70),
    "memory-limited": (0.72, 0.42),
    "latency reserve": (0.55, 0.50),
}
for label, (x0, y0) in points.items():
    ax.scatter([x0], [y0], s=90)
    ax.text(x0+0.02, y0+0.02, label, fontsize=10)

savefig("43_resource_surface.png")


## 9. Batch admission controller

Admission control is where resource allocation becomes operational.

The controller can reduce admitted batch size, shorten prefixes, defer low-value suffixes, or hold requests in the queue.


In [ ]:
# 43_batch_admission_control.png

fig, ax = plt.subplots(figsize=(12.7, 5.4))
ax.axis("off")
ax.set_xlim(0, 12.7)
ax.set_ylim(0, 5.4)

rounded_box(ax, (0.55, 3.05), 1.65, 0.75, "Incoming\nrequests", fontsize=11)
rounded_box(ax, (3.1, 3.05), 1.35, 0.75, "Queue", fontsize=12)
rounded_box(ax, (5.4, 2.9), 1.95, 1.05, "Admission /\nallocation\ncontroller", fontsize=11)
rounded_box(ax, (8.2, 3.05), 1.65, 0.75, "Admitted\nbatch", fontsize=11)
rounded_box(ax, (10.7, 3.05), 1.45, 0.75, "Target\nverify", fontsize=11)

arrow(ax, (2.25, 3.42), (3.05, 3.42))
arrow(ax, (4.5, 3.42), (5.35, 3.42))
arrow(ax, (7.4, 3.42), (8.15, 3.42))
arrow(ax, (9.9, 3.42), (10.65, 3.42))

signals = [
    ("compute\nbudget", 3.6, 1.05, 1.35, 0.62),
    ("memory\nbudget", 5.25, 1.05, 1.35, 0.62),
    ("latency\ntarget", 6.9, 1.05, 1.35, 0.62),
    ("reserve\ncapacity", 8.55, 1.05, 1.35, 0.62),
]
for label, x, y, w, h in signals:
    rounded_box(ax, (x, y), w, h, label, fontsize=10)
    arrow(ax, (x+w/2, y+h+0.05), (6.38, 2.85), lw=1.4)

rounded_box(ax, (3.4, 4.35), 5.9, 0.58, "Admission depends on the current resource budget.", fontsize=12)
ax.set_title("Batch admission control under resource constraints", fontsize=22, pad=14)
savefig("43_batch_admission_control.png")


## 10. Latency-memory tradeoff

The same prefix schedule can have different memory and latency costs under different regimes.

Notebook 43 treats these tradeoffs as policy inputs rather than after-the-fact measurements.


In [ ]:
# 43_latency_memory_tradeoff.png

ell = np.arange(1, 9)
params = {
    "balanced": (0.82, 0.80),
    "compute-limited": (0.78, 0.76),
    "memory-limited": (0.55, 0.92),
    "latency-limited": (1.05, 0.58),
}

fig, ax = plt.subplots(figsize=(9.5, 6.1))
for name, (lat_w, mem_w) in params.items():
    latency = lat_w * (0.25 + 0.13*ell + 0.018*ell**2)
    memory = mem_w * (0.28 + 0.18*ell + 0.014*ell**2)
    score = (1 - np.exp(-0.75*ell)) / (0.75 + 0.18*latency + 0.12*memory)
    best = np.argmax(score)
    ax.plot(memory, latency, marker="o", label=name)
    ax.scatter([memory[best]], [latency[best]], s=90)
    ax.text(memory[best]+0.04, latency[best], f"$\\ell^*={ell[best]}$", fontsize=10)

ax.set_xlabel("memory pressure proxy")
ax.set_ylabel("latency proxy")
ax.set_title("Latency-memory tradeoff depends on operating regime", fontsize=22)
ax.grid(True, alpha=0.3)
ax.legend()
savefig("43_latency_memory_tradeoff.png")


## Key equations

Resource budget:

\[
B_t=(B_{\rm compute},B_{\rm memory},B_{\rm verify},B_{\rm reserve})
\]

Regime-conditioned policy:

\[
B_t=b(\rho_t,s_t)
\]

Allocation:

\[
a_t=\pi(\rho_t,B_t,s_t)
\]

Feasible allocation set:

\[
\mathcal{A}_t
=
\{a:\ C(a)\le B_{\rm compute},\ M(a)\le B_{\rm memory},\ L(a)\le B_{\rm latency}\}
\]

Adaptive objective:

\[
a_t^*
=
\arg\max_{a\in\mathcal{A}_t}
\Theta(a;\rho_t)
\]

Feedback update:

\[
s_{t+1}=h(s_t,a_t^*,\Theta_t,U_t,Q_t)
\]

where \(U_t\) is utilization and \(Q_t\) is queue state.


## Engineering summary

Notebook 43 v2 reframes resource allocation as a controller.

Operating regimes from Notebook 37 are not endpoints; they are inputs to a resource policy. The controller partitions compute, memory, verification, reserve, and latency headroom across active requests, then updates allocation using observed serving state.

> **Operating regimes specify adaptive resource allocation.**


In [ ]:
# Save manifest

manifest = {
    "notebook": "43_adaptive_resource_allocation_v2.ipynb",
    "title": "Adaptive Resource Allocation",
    "engineering_statement": "Operating regimes specify adaptive resource allocation.",
    "figures": [
        "43_repository_arc.png",
        "43_controller_architecture.png",
        "43_resource_flow.png",
        "43_budget_partition.png",
        "43_policy_curves.png",
        "43_policy_surface.png",
        "43_phase_map.png",
        "43_resource_surface.png",
        "43_batch_admission_control.png",
        "43_latency_memory_tradeoff.png",
    ],
    "previous_notebook": "37_operating_regimes.ipynb",
    "next_notebook": "49_adaptive_infrastructure.ipynb",
}
(RESULTS / "43_adaptive_resource_allocation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## Download artifacts

Run the final cell to package Notebook 43 v2 artifacts.

The archive includes:

- notebook
- generated figures
- generated JSON outputs
- manifest


In [ ]:
# Package Notebook 43 v2 artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_adaptive_resource_allocation"
SITE = REPO_ROOT / "site"

RESULTS.mkdir(parents=True, exist_ok=True)
zip_path = RESULTS / "43_adaptive_resource_allocation_v2_artifacts.zip"

notebook_candidates = [
    NOTEBOOKS / "43_adaptive_resource_allocation_v2.ipynb",
    Path.cwd() / "43_adaptive_resource_allocation_v2.ipynb",
    Path.cwd() / "43_resource_allocation.ipynb",
]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)

for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
